In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes pandas tqdm

from google.colab import drive
# Ini akan memunculkan pop-up minta izin akses ke Google Drive kamu
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import pandas as pd
import json

OUT_DIR = "/content/drive/MyDrive/stem_err_id"

# 1. Baca data soal yang ada teks lengkapnya (dari Notebook 01)
df_soal = pd.read_csv(f"{OUT_DIR}/sains_bersih.csv")

# 2. Baca data hasil ujian AI dari Notebook 02 (Misal kita perbaiki TinyLlama)
file_hasil = f"{OUT_DIR}/arsip_pilot/hasil_TinyLlama-1.1B-Chat-v1.0_0shot_pilot.csv"
df_hasil = pd.read_csv(file_hasil)

# 3. Gabungkan hasil ujian dengan teks soal berdasarkan kolom 'id'
df_gabungan = pd.merge(df_hasil, df_soal[['id', 'soal', 'opsi_list']], on='id', how='left')

# 4. Saring HANYA soal-soal yang dijawab SALAH oleh AI (kolom benar == 0)
df_error = df_gabungan[df_gabungan['benar'] == 0].copy()

# 5. Rapikan format opsi_list (karena aslinya tersimpan sebagai teks JSON)
df_error['opsi_list'] = df_error['opsi_list'].apply(json.loads)

print(f"Total soal yang salah dan siap di-voting ulang: {len(df_error)} soal")
display(df_error[['id', 'mapel', 'pred', 'kunci', 'soal']].head(3))

Total soal yang salah dan siap di-voting ulang: 78 soal


,id,mapel,pred,kunci,soal
1,7009,Fisika,D,E,Pernyataan yang menggambarkan dan menjelaskan ...
2,3389,Biologi,B,C,. Buaya di masukkan ke dalam kelompok hewan te...
4,2941,Biologi,B,D,Hasil dari pembelahan mitosis adalah ....\n


In [ ]:
import torch
import re
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Panggil model yang sama dengan yang diuji temanmu
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Memuat model {model_id} untuk mitigasi...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)

# Fungsi untuk bertanya ke AI
def tanya_ai(soal, pilihan_jawaban):
    HURUF = ["A", "B", "C", "D", "E"]
    opsi_teks = "\n".join(f"{h}. {o}" for h, o in zip(HURUF, pilihan_jawaban))

    prompt = f"Ini adalah soal pilihan ganda. Pilihlah satu jawaban yang paling tepat.\n\n{soal}\n{opsi_teks}\nJawaban:"
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=True, temperature=0.7)

    jawaban_mentah = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r'\b([A-E])\b', jawaban_mentah.upper())
    return match.group(1) if match else jawaban_mentah[:1].upper()

# Fungsi Mitigasi: Voting 5 Kali
def tanya_ai_voting(soal, pilihan_jawaban, jumlah_voting=5):
    jawaban_terkumpul = []
    for _ in range(jumlah_voting):
        try:
            jwb = tanya_ai(soal, pilihan_jawaban)
        except Exception:
            jwb = "ERROR"
        jawaban_terkumpul.append(jwb)

    # Ambil suara terbanyak
    hitung_suara = Counter(jawaban_terkumpul)
    jawaban_final = hitung_suara.most_common(1)[0][0]
    return jawaban_final, str(jawaban_terkumpul)

# Eksekusi ke soal yang salah
tqdm.pandas(desc="AI Sedang Voting")
hasil = df_error.progress_apply(lambda row: tanya_ai_voting(row['soal'], row['opsi_list']), axis=1)

df_error['jawaban_voting'] = [h[0] for h in hasil]
df_error['riwayat_voting'] = [h[1] for h in hasil]
df_error['benar_setelah_voting'] = (df_error['jawaban_voting'] == df_error['kunci']).astype(int)

# Simpan hasil mitigasimu ke Drive
df_error.to_csv("/content/hasil_mitigasi_voting.csv", index=False)

# Hitung keberhasilan mitigasi
soal_diperbaiki = df_error['benar_setelah_voting'].sum()
persentase_naik = (soal_diperbaiki / len(df_error)) * 100
print(f"\nMITIGASI SELESAI!")
print(f"Dari {len(df_error)} soal yang tadinya salah, AI berhasil menjawab benar {soal_diperbaiki} soal setelah sistem voting.")
print(f"Akurasi berhasil ditingkatkan sebesar {persentase_naik:.1f}% untuk kelompok soal error ini!")

Memuat model TinyLlama/TinyLlama-1.1B-Chat-v1.0 untuk mitigasi...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

AI Sedang Voting:   0%|          | 0/78 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docu


MITIGASI SELESAI!
Dari 78 soal yang tadinya salah, AI berhasil menjawab benar 5 soal setelah sistem voting.
Akurasi berhasil ditingkatkan sebesar 6.4% untuk kelompok soal error ini!


In [ ]:
import gc
import torch
import re
import json
import pandas as pd
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

OUT_DIR = "/content/drive/MyDrive/stem_err_id"
df_soal = pd.read_csv(f"{OUT_DIR}/sains_bersih.csv")

# Daftar model tambahan dari hasil pilot temanmu
model_tambahan = [
    ("Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "hasil_Qwen2.5-3B_0shot_pilot.csv"),
    ("Phi-3.5-mini", "microsoft/Phi-3.5-mini-instruct", "hasil_Phi-3.5-mini-instruct_0shot_pilot.csv"),
]

# Bersihkan sisa memori TinyLlama tadi
if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

rekap_mitigasi = []

# Masukkan hasil TinyLlama yang sudah kamu run tadi ke rekap
if 'df_error' in globals():
    jml_salah_tiny = len(df_error)
    jml_bener_tiny = df_error['benar_setelah_voting'].sum()
    rekap_mitigasi.append({
        "Model": "TinyLlama-1.1B",
        "Akurasi_Awal (%)": 100 - jml_salah_tiny,
        "Jumlah_Soal_Salah": jml_salah_tiny,
        "Berhasil_Diperbaiki": jml_bener_tiny,
        "Efektivitas_Mitigasi (%)": round((jml_bener_tiny / jml_salah_tiny) * 100, 2),
        "Akurasi_Akhir_Total (%)": (100 - jml_salah_tiny) + jml_bener_tiny
    })

for nama_pendek, hf_id, nama_file_csv in model_tambahan:
    print(f"\n==========================================")
    print(f"Memproses Mitigasi: {nama_pendek}...")

    # Cari file di arsip_pilot atau folder utama
    path_csv = f"{OUT_DIR}/arsip_pilot/{nama_file_csv}"
    try:
        df_h = pd.read_csv(path_csv)
    except FileNotFoundError:
        df_h = pd.read_csv(f"{OUT_DIR}/{nama_file_csv}")

    df_gab = pd.merge(df_h, df_soal[['id', 'soal', 'opsi_list']], on='id', how='left')
    df_err_model = df_gab[df_gab['benar'] == 0].copy()
    df_err_model['opsi_list'] = df_err_model['opsi_list'].apply(json.loads)

    print(f"Total soal salah di {nama_pendek}: {len(df_err_model)} soal")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float16, device_map="auto")

    def tanya_ai_lokal(soal, pilihan_jawaban):
        HURUF = ["A", "B", "C", "D", "E"]
        opsi_teks = "\n".join(f"{h}. {o}" for h, o in zip(HURUF, pilihan_jawaban))
        prompt = f"Ini adalah soal pilihan ganda. Pilihlah satu jawaban yang paling tepat.\n\n{soal}\n{opsi_teks}\nJawaban:"
        messages = [{"role": "user", "content": prompt}]
        try:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            text = prompt + " "
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=5, do_sample=True, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
        jawaban_mentah = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        match = re.search(r'\b([A-E])\b', jawaban_mentah.upper())
        return match.group(1) if match else (jawaban_mentah[:1].upper() if jawaban_mentah else "X")

    jawaban_voting_list = []
    riwayat_voting_list = []

    for _, row in tqdm(df_err_model.iterrows(), total=len(df_err_model), desc=f"Voting {nama_pendek}"):
        terkumpul = []
        for _ in range(5):
            try:
                jwb = tanya_ai_lokal(row['soal'], row['opsi_list'])
            except Exception:
                jwb = "ERROR"
            terkumpul.append(jwb)
        jawaban_final = Counter(terkumpul).most_common(1)[0][0]
        jawaban_voting_list.append(jawaban_final)
        riwayat_voting_list.append(str(terkumpul))

    df_err_model['jawaban_voting'] = jawaban_voting_list
    df_err_model['riwayat_voting'] = riwayat_voting_list
    df_err_model['benar_setelah_voting'] = (df_err_model['jawaban_voting'] == df_err_model['kunci']).astype(int)

    # Simpan ke lokal Colab
    df_err_model.to_csv(f"/content/hasil_mitigasi_{nama_pendek}.csv", index=False)

    jml_salah = len(df_err_model)
    jml_diperbaiki = df_err_model['benar_setelah_voting'].sum()
    akurasi_awal = 100 - jml_salah

    rekap_mitigasi.append({
        "Model": nama_pendek,
        "Akurasi_Awal (%)": akurasi_awal,
        "Jumlah_Soal_Salah": jml_salah,
        "Berhasil_Diperbaiki": jml_diperbaiki,
        "Efektivitas_Mitigasi (%)": round((jml_diperbaiki / jml_salah) * 100, 2),
        "Akurasi_Akhir_Total (%)": akurasi_awal + jml_diperbaiki
    })

    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

df_rekap_mitigasi = pd.DataFrame(rekap_mitigasi)
df_rekap_mitigasi.to_csv("/content/rekap_mitigasi_semua_model.csv", index=False)
print("\n🎉 REKAPITULASI MITIGASI SELURUH MODEL SELESAI!")
display(df_rekap_mitigasi)


Memproses Mitigasi: Qwen2.5-3B...
Total soal salah di Qwen2.5-3B: 50 soal


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]



Voting Phi-3.5-mini:   0%|          | 0/62 [00:00<?, ?it/s]

Voting Phi-3.5-mini:   2%|▏         | 1/62 [00:01<01:52,  1.85s/it]

Voting Phi-3.5-mini:   3%|▎         | 2/62 [00:03<01:26,  1.44s/it]

Voting Phi-3.5-mini:   5%|▍         | 3/62 [00:04<01:17,  1.31s/it]

Voting Phi-3.5-mini:   6%|▋         | 4/62 [00:05<01:12,  1.25s/it]

Voting Phi-3.5-mini:   8%|▊         | 5/62 [00:06<01:15,  1.32s/it]

Voting Phi-3.5-mini:  10%|▉         | 6/62 [00:08<01:16,  1.37s/it]

Voting Phi-3.5-mini:  11%|█▏        | 7/62 [00:09<01:11,  1.31s/it]

Voting Phi-3.5-mini:  13%|█▎        | 8/62 [00:10<01:11,  1.32s/it]

Voting Phi-3.5-mini:  15%|█▍        | 9/62 [00:11<01:07,  1.28s/it]

Voting Phi-3.5-mini:  16%|█▌        | 10/62 [00:13<01:07,  1.31s/it]

Voting Phi-3.5-mini:  18%|█▊        | 11/62 [00:14<01:08,  1.33s/it]

Voting Phi-3.5-mini:  19%|█▉        | 12/62 [00:15<01:05,  1.30s/it]

Voting Phi-3.5-mini:  21%|██        | 13/62 [00:17<01:02,  1.28s/it]

Voting Phi-3.5-mini:  23%|██▎       


🎉 REKAPITULASI MITIGASI SELURUH MODEL SELESAI!


,Model,Akurasi_Awal (%),Jumlah_Soal_Salah,Berhasil_Diperbaiki,Efektivitas_Mitigasi (%),Akurasi_Akhir_Total (%)
0,TinyLlama-1.1B,22,78,5,6.41,27
1,Qwen2.5-3B,50,50,1,2.00,51
2,Phi-3.5-mini,38,62,8,12.90,46
